In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import math


In [3]:
text = """
artificial intelligence is transforming modern society
it is used in healthcare finance education and transportation
machine learning allows systems to improve automatically with experience
data plays a critical role in training intelligent systems
large datasets help models learn complex patterns
deep learning uses multi layer neural networks
natural language processing helps computers understand human language
transformer models changed the field of nlp
attention allows the model to focus on relevant context
"""


In [5]:
words = text.lower().split()
vocab = sorted(set(words))

word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

vocab_size = len(vocab)


In [9]:
sequence_length = 4
X = []
Y = []

for i in range(len(words) - sequence_length):
    X.append([word_to_idx[w] for w in words[i:i+sequence_length]])
    Y.append(word_to_idx[words[i+sequence_length]])

X = torch.tensor(X)
Y = torch.tensor(Y)


lstm


In [6]:
class LSTMTextGenerator(nn.Module):
    def __init__(self, vocab_size, embed_size=50, hidden_size=100):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]      # last time step
        out = self.fc(out)
        return out


In [7]:
model = LSTMTextGenerator(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [10]:
epochs = 300

for epoch in range(epochs):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, Y)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 4.1210
Epoch 50, Loss: 1.7286
Epoch 100, Loss: 0.1050
Epoch 150, Loss: 0.0338
Epoch 200, Loss: 0.0184
Epoch 250, Loss: 0.0119


In [11]:
def generate_text(seed_text, num_words=10):
    model.eval()
    words_list = seed_text.lower().split()

    for _ in range(num_words):
        input_seq = words_list[-sequence_length:]
        input_idx = torch.tensor([[word_to_idx[w] for w in input_seq]])

        with torch.no_grad():
            prediction = model(input_idx)
            next_word_idx = torch.argmax(prediction).item()

        words_list.append(idx_to_word[next_word_idx])

    return " ".join(words_list)


In [12]:
seed = "artificial intelligence is"
print(generate_text(seed))


artificial intelligence is modern society it is used in healthcare finance education and


In [13]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [14]:
class TransformerTextGen(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.fc(x)


In [15]:
model = TransformerTextGen(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 300

for epoch in range(epochs):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, Y)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 0, Loss: 4.2093
Epoch 50, Loss: 0.8581
Epoch 100, Loss: 0.1435
Epoch 150, Loss: 0.0502
Epoch 200, Loss: 0.0283
Epoch 250, Loss: 0.0187


In [16]:
def generate_text(seed_text, num_words=10):
    model.eval()
    words_list = seed_text.lower().split()

    for _ in range(num_words):
        input_seq = words_list[-sequence_length:]
        input_idx = torch.tensor([[word_to_idx[w] for w in input_seq]])

        with torch.no_grad():
            prediction = model(input_idx)
            next_word_idx = torch.argmax(prediction).item()

        words_list.append(idx_to_word[next_word_idx])

    return " ".join(words_list)


In [17]:
seed = "artificial intelligence is transforming"
print(generate_text(seed))


artificial intelligence is transforming modern society it is used in healthcare finance education and
